# London Housing Market Analysis (1995-2020)

**Research question:** How have London house prices evolved between 1995 and 2020, and how do price levels differ across boroughs?

**Data:** *London Housing Data* (UK Land Registry / ONS / Met Police) - 13,549 monthly records for 45 London areas, 1995-01 to 2020-01.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import os

df = pd.read_csv("data/London Housing Data.csv")
print(df.shape)
df.head()

## 1. Data cleaning

In [ ]:
df["date"] = pd.to_datetime(df["date"], format="%d/%m/%Y")
df["year"] = df["date"].dt.year
df["month"] = df["date"].dt.month
crime_df = df.dropna(subset=["no_of_crimes"]).copy()
print("rows with crime data:", len(crime_df))

## 2. London-wide price trend

In [ ]:
london_monthly = df.groupby("date")["average_price"].mean().resample("ME").mean().dropna()
print("1995:", round(london_monthly.iloc[0]), "| 2020:", round(london_monthly.iloc[-1]))

## 3. Visualizations

In [ ]:
plt.rcParams.update({"figure.dpi": 150, "axes.spines.top": False, "axes.spines.right": False})
os.makedirs("visualizations", exist_ok=True)

# Fig 1: trend
fig, ax = plt.subplots(figsize=(9, 4.8))
ax.plot(london_monthly.index, london_monthly.values, color="#1f77b4", lw=2)
ax.set_title("Average House Price in London, 1995 - 2020")
ax.set_ylabel("Average price (GBP)")
ax.grid(True, alpha=0.3)
fig.savefig("visualizations/fig1_london_price_trend.png", bbox_inches="tight")

In [ ]:
# Fig 2: top 10 boroughs (2018)
area_latest = df[df["year"] == 2018].groupby("area")["average_price"].mean().sort_values(ascending=False)
top10 = area_latest.head(10).iloc[::-1]
fig, ax = plt.subplots(figsize=(9, 4.8))
ax.barh(top10.index, top10.values / 1000, color="#2ca02c", alpha=0.85)
ax.set_title("Top 10 London Boroughs by Average House Price (2018)")
ax.set_xlabel("Average price (thousand GBP)")
fig.savefig("visualizations/fig2_top10_areas.png", bbox_inches="tight")

In [ ]:
# Fig 3: price vs crime
area_stats = crime_df.groupby("area").agg(avg_price=("average_price", "mean"), avg_crime=("no_of_crimes", "mean"), total_sold=("houses_sold", "sum"))
fig, ax = plt.subplots(figsize=(9, 4.8))
ax.scatter(area_stats["avg_crime"], area_stats["avg_price"] / 1000, s=20, alpha=0.7, color="#ff7f0e")
ax.set_title("House Price vs. Crime Rate across London Boroughs")
ax.set_xlabel("Average monthly crimes")
ax.set_ylabel("Average house price (thousand GBP)")
fig.savefig("visualizations/fig3_price_vs_crime.png", bbox_inches="tight")
print("Figures saved.")

## 4. Key findings
1. Average price rose from ~GBP 76k (1995) to ~GBP 463k (2020), **+511%**.
2. A clear dip around the **2008 financial crisis**, then recovery to a peak annual average of ~GBP 470k (2017).
3. **Kensington and Chelsea** is the most expensive borough (2018, ~GBP 1.36M); **North East** the least (~GBP 128k).
4. Borough-level price-crime correlation is mild (**r ~ 0.29**).

**Limitation:** ~45% of crime records are missing; the price-crime link is based on a subset and is not causal.